# Phase 2 — Final Version (Complete & Self-Contained)

**Structure:**
- Cell 0: Auth (run once per session)
- Cell 1: Setup (run once per session)
- Cell 2: Static extraction — DEM + Land Use (run ONCE EVER, auto-skipped after)
- Cell 3: Weekly loop — S5P + ERA5-Land (run repeatedly, change LOOP_START + N_WEEKS)

**All outputs go to:** `MyDrive/Iran_AirPollution/`

## Cell 0 — Authentication (Once Per Session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'earthengine-api', '--quiet'])
import ee
ee.Authenticate()
ee.Initialize()
print('Drive mounted. GEE authenticated.')

## Cell 1 — Setup (Once Per Session)

In [ ]:
import os, time, warnings
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
DRIVE_BASE         = '/content/drive/MyDrive/Iran_AirPollution'
MASTER_GROUND_FILE = f'{DRIVE_BASE}/master_ground.csv'
OUTPUT_FILE        = f'{DRIVE_BASE}/master_satellite.csv'
STATIC_FILE        = f'{DRIVE_BASE}/static_features.csv'
os.makedirs(DRIVE_BASE, exist_ok=True)

# ── Scales (verified from official GEE Data Catalog) ───────────────────────
SCALE_S5P       = 1113   # Sentinel-5P L3 pixel size: 1113.2m
SCALE_ERA5_LAND = 11132  # ERA5-Land: 0.1 degree ~ 11,132m
SCALE_COP_DEM   = 30     # Copernicus DEM GLO-30: 30m
SCALE_LANDUSE   = 10     # ESA WorldCover v200: 10m

# ── Load stations ──────────────────────────────────────────────────────────
assert os.path.exists(MASTER_GROUND_FILE), \
    f'Upload master_ground.csv to: {DRIVE_BASE}'

master_ground = pd.read_csv(MASTER_GROUND_FILE)
assert {'station_id','latitude','longitude'}.issubset(master_ground.columns), \
    'master_ground.csv must have: station_id, latitude, longitude'

stations = (
    master_ground[['station_id','latitude','longitude']]
    .drop_duplicates(subset='station_id')
    .dropna(subset=['latitude','longitude'])
    .reset_index(drop=True)
)

# ── Build GEE FeatureCollection ────────────────────────────────────────────
station_fc = ee.FeatureCollection([
    ee.Feature(
        ee.Geometry.Point([float(r['longitude']), float(r['latitude'])]),
        {'station_id': str(r['station_id'])}
    )
    for _, r in stations.iterrows()
])

# ── S5P collections ────────────────────────────────────────────────────────
S5P_COLLECTIONS = {
    'NO2_sat': ('COPERNICUS/S5P/OFFL/L3_NO2',
                'tropospheric_NO2_column_number_density'),
    'CO_sat' : ('COPERNICUS/S5P/OFFL/L3_CO',
                'CO_column_number_density'),
    'O3_sat' : ('COPERNICUS/S5P/OFFL/L3_O3',
                'O3_column_number_density'),
    'SO2_sat': ('COPERNICUS/S5P/OFFL/L3_SO2',
                'SO2_column_number_density')
}

# ── ERA5-Land bands ────────────────────────────────────────────────────────
ERA5_LAND_BANDS = [
    'temperature_2m',           # K
    'dewpoint_temperature_2m',  # K
    'u_component_of_wind_10m',  # m/s
    'v_component_of_wind_10m',  # m/s
    'surface_pressure',         # Pa
    'total_precipitation_sum'   # m
]

# ── ESA WorldCover class labels ────────────────────────────────────────────
LANDUSE_LABELS = {
    10:'tree_cover', 20:'shrubland', 30:'grassland',
    40:'cropland',   50:'built_up',  60:'bare_desert',
    70:'snow_ice',   80:'water',     90:'wetland',
    95:'mangroves',  100:'moss_lichen'
}

# ── Status report ──────────────────────────────────────────────────────────
print(f'Stations loaded    : {len(stations)}')
print(f'GEE FC size        : {station_fc.size().getInfo()}')
print(f'Static file exists : {os.path.exists(STATIC_FILE)}')
print(f'Output file exists : {os.path.exists(OUTPUT_FILE)}')
if os.path.exists(OUTPUT_FILE):
    ex = pd.read_csv(OUTPUT_FILE)
    weeks = sorted(ex['week_start'].unique())
    print(f'Weeks in file      : {len(weeks)}')
    print(f'Last week done     : {weeks[-1]}')
print('\nSetup complete.')

## Cell 2 — Static Features: DEM + Land Use
**Run once ever. Auto-skipped if `static_features.csv` already exists.**

Sources:
- **Copernicus DEM GLO-30** — 30m, RMSE=1.68m (TanDEM-X, 2011-2015)
- **ESA WorldCover v200** — 10m, accuracy=76.7% (Sentinel-1+2, 2021)

In [ ]:
if os.path.exists(STATIC_FILE):
    static_df = pd.read_csv(STATIC_FILE)
    print(f'static_features.csv already exists ({len(static_df)} stations). Skipping.')
    print(static_df[['station_id','elevation_m','landuse_label']].head())

else:
    print('Extracting static features (this runs only once)...')

    # ── 1. Copernicus DEM GLO-30 ───────────────────────────────────────────
    print('\n[1/2] Copernicus DEM GLO-30...')
    try:
        cop_dem   = ee.ImageCollection('COPERNICUS/DEM/GLO30') \
                      .select('DEM').mosaic()
        dem_samp  = cop_dem.reduceRegions(
            collection=station_fc,
            reducer=ee.Reducer.mean(),
            scale=SCALE_COP_DEM, tileScale=4
        )
        dem_rows  = dem_samp.getInfo()['features']
        dem_df    = pd.DataFrame([
            {'station_id' : f['properties'].get('station_id'),
             'elevation_m': round(float(f['properties']['mean']), 2)
                            if f['properties'].get('mean') is not None
                            else np.nan,
             'dem_source' : 'Copernicus_GLO30'}
            for f in dem_rows
        ])

        # Fallback to SRTM for any stations missing in Copernicus
        missing_mask = dem_df['elevation_m'].isna()
        n_missing    = missing_mask.sum()
        if n_missing > 0:
            print(f'  {n_missing} stations missing in Copernicus -> SRTM fallback')
            missing_ids  = dem_df[missing_mask]['station_id'].tolist()
            missing_rows = stations[stations['station_id'].isin(missing_ids)]
            fallback_fc  = ee.FeatureCollection([
                ee.Feature(
                    ee.Geometry.Point([float(r['longitude']),
                                       float(r['latitude'])]),
                    {'station_id': str(r['station_id'])}
                )
                for _, r in missing_rows.iterrows()
            ])
            srtm_samp = ee.Image('USGS/SRTMGL1_003').select('elevation') \
                          .reduceRegions(
                              collection=fallback_fc,
                              reducer=ee.Reducer.mean(),
                              scale=30, tileScale=4
                          )
            for f in srtm_samp.getInfo()['features']:
                sid = f['properties'].get('station_id')
                val = f['properties'].get('mean')
                idx = dem_df[dem_df['station_id'] == sid].index
                if len(idx) and val is not None:
                    dem_df.loc[idx[0], 'elevation_m'] = round(float(val), 2)
                    dem_df.loc[idx[0], 'dem_source']  = 'SRTM_fallback'

        print(f'  Done. Elevation range: '
              f'{dem_df["elevation_m"].min():.1f}m '
              f'to {dem_df["elevation_m"].max():.1f}m')

    except Exception as e:
        print(f'  Copernicus DEM failed: {e}')
        print('  Using SRTM for all stations...')
        srtm_samp = ee.Image('USGS/SRTMGL1_003').select('elevation') \
                      .reduceRegions(
                          collection=station_fc,
                          reducer=ee.Reducer.mean(),
                          scale=30, tileScale=4
                      )
        dem_df = pd.DataFrame([
            {'station_id' : f['properties'].get('station_id'),
             'elevation_m': round(float(f['properties']['mean']), 2)
                            if f['properties'].get('mean') is not None
                            else np.nan,
             'dem_source' : 'SRTM_fallback'}
            for f in srtm_samp.getInfo()['features']
        ])

    # ── 2. ESA WorldCover v200 ─────────────────────────────────────────────
    print('\n[2/2] ESA WorldCover v200 (10m)...')
    try:
        worldcover = ee.ImageCollection('ESA/WorldCover/v200') \
                       .first().select('Map')
        lc_samp    = worldcover.reduceRegions(
            collection=station_fc,
            reducer=ee.Reducer.mode(),
            scale=SCALE_LANDUSE, tileScale=4
        )
        lc_rows = lc_samp.getInfo()['features']
        lc_df   = pd.DataFrame([
            {'station_id'   : f['properties'].get('station_id'),
             'landuse_class': int(f['properties']['mode'])
                              if f['properties'].get('mode') is not None
                              else np.nan}
            for f in lc_rows
        ])
        lc_df['landuse_label'] = lc_df['landuse_class'].map(
            lambda x: LANDUSE_LABELS.get(int(x), 'unknown')
                      if pd.notna(x) else 'unknown'
        )
        # One-hot encode for ML
        for cls_val, cls_name in LANDUSE_LABELS.items():
            lc_df[f'lu_{cls_name}'] = \
                (lc_df['landuse_class'] == cls_val).astype(int)

        print('  Land use distribution:')
        print(lc_df['landuse_label'].value_counts().to_string())

    except Exception as e:
        print(f'  WorldCover failed: {e} -> filling with unknown')
        lc_df = pd.DataFrame({
            'station_id'   : stations['station_id'],
            'landuse_class': np.nan,
            'landuse_label': 'unknown'
        })
        for cls_name in LANDUSE_LABELS.values():
            lc_df[f'lu_{cls_name}'] = 0

    # ── Merge and save ─────────────────────────────────────────────────────
    static_df = dem_df.merge(lc_df, on='station_id', how='left')
    static_df.to_csv(STATIC_FILE, index=False, encoding='utf-8-sig')
    print(f'\nstatic_features.csv saved -> {STATIC_FILE}')
    print(static_df[['station_id','elevation_m','dem_source','landuse_label']].head(10))

## Cell 3 — Weekly Loop: S5P + ERA5-Land

**Change `LOOP_START` and `N_WEEKS`, then run.**

Sources:
- **Sentinel-5P OFFL L3** — 1113.2m pixel, daily global, QA-filtered
- **ERA5-Land DAILY_AGGR** — 0.1° (~11,132m), no gaps, 1950–present

In [ ]:
# ██████████████████████████████████████████████████████
# !! CHANGE THESE TWO VALUES, THEN RUN !!
LOOP_START = '2018-10-01'   # Monday, YYYY-MM-DD
N_WEEKS    = 10             # weeks to process this run
# ██████████████████████████████████████████████████████

# ── Validate ───────────────────────────────────────────────────────────────
loop_start_dt = datetime.strptime(LOOP_START, '%Y-%m-%d')
assert loop_start_dt >= datetime(2018, 10, 1), \
    'S5P data starts Oct 2018. Use a later date.'
assert os.path.exists(STATIC_FILE), \
    'static_features.csv not found. Run Cell 2 first.'

# ── Load static & done weeks ───────────────────────────────────────────────
static_df = pd.read_csv(STATIC_FILE)
lu_cols   = [c for c in static_df.columns if c.startswith('lu_')]
static_join_cols = [
    c for c in
    ['station_id','elevation_m','dem_source','landuse_class','landuse_label']
    + lu_cols
    if c in static_df.columns
]

done_weeks = set()
if os.path.exists(OUTPUT_FILE):
    done_weeks = set(
        pd.read_csv(OUTPUT_FILE)['week_start'].astype(str).unique()
    )

final_col_order = [
    'station_id', 'week_start',
    'NO2_sat', 'CO_sat', 'O3_sat', 'SO2_sat',
    'temp_c', 'humidity_pct', 'wind_speed', 'wind_dir',
    'pressure_hpa', 'precip_mm',
    'elevation_m', 'dem_source', 'landuse_class', 'landuse_label'
] + lu_cols

print(f'Loop start    : {LOOP_START}')
print(f'Weeks to run  : {N_WEEKS}')
print(f'Already done  : {len(done_weeks)} weeks')
print(f'Static loaded : {len(static_df)} stations')
print('-' * 55)

# ── Main loop ──────────────────────────────────────────────────────────────
for i in range(N_WEEKS):
    ws_dt = loop_start_dt + timedelta(weeks=i)
    we_dt = ws_dt + timedelta(days=7)
    WS    = ws_dt.strftime('%Y-%m-%d')
    WE    = we_dt.strftime('%Y-%m-%d')

    print(f'\n[{i+1:>3}/{N_WEEKS}] {WS}')

    if WS in done_weeks:
        print('  Already done -> skipping')
        continue

    week_df = stations[['station_id']].copy()
    week_df['week_start'] = WS

    # ── S5P extraction ────────────────────────────────────────────────────
    for col_name, (coll_id, band) in S5P_COLLECTIONS.items():
        time.sleep(1)
        try:
            col = ee.ImageCollection(coll_id) \
                    .filterDate(WS, WE).select(band)
            n   = col.size().getInfo()
            if n == 0:
                week_df[col_name] = np.nan
                print(f'  {col_name:<10}: no images -> NaN')
                continue
            samp = col.mean().rename('value').reduceRegions(
                collection=station_fc,
                reducer=ee.Reducer.mean(),
                scale=SCALE_S5P, tileScale=4
            )
            rows    = samp.getInfo()['features']
            tmp_df  = pd.DataFrame([
                {'station_id': f['properties'].get('station_id'),
                 col_name    : f['properties'].get('mean')}
                for f in rows
            ])
            week_df = week_df.merge(tmp_df, on='station_id', how='left')
            n_ok    = week_df[col_name].notna().sum()
            print(f'  {col_name:<10}: {n} imgs | {n_ok}/{len(week_df)} valid')
        except Exception as e:
            week_df[col_name] = np.nan
            print(f'  {col_name:<10}: ERROR - {e}')

    # ── ERA5-Land extraction ───────────────────────────────────────────────
    time.sleep(1)
    try:
        era5_col = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR') \
                     .filterDate(WS, WE).select(ERA5_LAND_BANDS)
        n_era5   = era5_col.size().getInfo()

        if n_era5 == 0:
            print(f'  ERA5-Land  : no images -> NaN')
            for c in ['temp_c','humidity_pct','wind_speed',
                      'wind_dir','pressure_hpa','precip_mm']:
                week_df[c] = np.nan
        else:
            samp    = era5_col.mean().reduceRegions(
                collection=station_fc,
                reducer=ee.Reducer.mean(),
                scale=SCALE_ERA5_LAND, tileScale=4
            )
            rows    = samp.getInfo()['features']
            e5_df   = pd.DataFrame([
                {'station_id'  : f['properties'].get('station_id'),
                 'temp_k'      : f['properties'].get('temperature_2m'),
                 'dewpoint_k'  : f['properties'].get('dewpoint_temperature_2m'),
                 'wind_u'      : f['properties'].get('u_component_of_wind_10m'),
                 'wind_v'      : f['properties'].get('v_component_of_wind_10m'),
                 'pressure_pa' : f['properties'].get('surface_pressure'),
                 'precip_m'    : f['properties'].get('total_precipitation_sum')}
                for f in rows
            ])
            # Derive physical features
            t  = e5_df['temp_k']     - 273.15
            td = e5_df['dewpoint_k'] - 273.15
            e5_df['temp_c']       = t
            e5_df['wind_speed']   = np.sqrt(
                e5_df['wind_u']**2 + e5_df['wind_v']**2)
            e5_df['wind_dir']     = (
                np.degrees(np.arctan2(
                    e5_df['wind_u'], e5_df['wind_v'])) + 180) % 360
            e5_df['humidity_pct'] = (100 *
                np.exp((17.625*td)/(243.04+td)) /
                np.exp((17.625*t) /(243.04+t))).clip(0, 100)
            e5_df['pressure_hpa'] = e5_df['pressure_pa'] / 100
            e5_df['precip_mm']    = e5_df['precip_m'] * 1000
            e5_df = e5_df[['station_id','temp_c','humidity_pct',
                            'wind_speed','wind_dir',
                            'pressure_hpa','precip_mm']]
            week_df = week_df.merge(e5_df, on='station_id', how='left')
            n_ok    = e5_df['temp_c'].notna().sum()
            print(f'  ERA5-Land  : {n_era5} imgs | {n_ok}/{len(e5_df)} valid')

    except Exception as e:
        print(f'  ERA5-Land  : ERROR - {e}')
        for c in ['temp_c','humidity_pct','wind_speed',
                  'wind_dir','pressure_hpa','precip_mm']:
            week_df[c] = np.nan

    # ── Join static features ───────────────────────────────────────────────
    week_df = week_df.merge(
        static_df[static_join_cols], on='station_id', how='left'
    )

    # ── Reorder columns ────────────────────────────────────────────────────
    ordered = [c for c in final_col_order if c in week_df.columns]
    week_df = week_df[ordered]

    # ── Append to master_satellite.csv ─────────────────────────────────────
    if os.path.exists(OUTPUT_FILE):
        week_df.to_csv(OUTPUT_FILE, mode='a', header=False,
                       index=False, encoding='utf-8-sig')
    else:
        week_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

    done_weeks.add(WS)
    print(f'  Saved. Total weeks in file: {len(done_weeks)}')

# ── End of loop summary ────────────────────────────────────────────────────
next_start = (loop_start_dt + timedelta(weeks=N_WEEKS)).strftime('%Y-%m-%d')
print('\n' + '=' * 55)
print(f'Done. {N_WEEKS} weeks processed.')
print(f'Total weeks in file: {len(done_weeks)}')
print(f'\n>>> Next run:')
print(f"LOOP_START = '{next_start}'")